# StyleGAN (simplified) — style-based synthesis with AdaIN

> Tutorial pair for [`stylegan.py`](stylegan.py).

## 1. Intuition
A vanilla generator pipes the latent code straight into the first layer, so all
factors of variation are tangled at the input. StyleGAN instead turns the latent
into a **style** that *modulates every layer*. It (1) maps $z$ to a disentangled
space $w$, (2) injects $w$ at each synthesis layer via **AdaIN**, and (3) adds
per-pixel **noise** for stochastic detail. Coarse layers control coarse
attributes, fine layers fine ones — giving the famous layer-wise control.

## 2. Concept (the slide)
- **Mapping network** $f:z\mapsto w$ — an MLP that disentangles the latent
  (after normalizing $z$ onto the hypersphere).
- **Learned constant input** — synthesis starts from a learned $4\times4$ tensor,
  *not* from $z$.
- **AdaIN** — each layer's activations are instance-normalized then re-styled by
  a scale/bias predicted from $w$.
- **Noise injection** — per-pixel Gaussian noise with a learned per-channel scale
  adds texture the style need not encode.
- The discriminator is ordinary. Toy data: tiny 1x16x16 images.

## 3. Math derivation — AdaIN and style-based synthesis

**Mapping network.** First normalize $z$ (PixelNorm onto the hypersphere),
$\hat z = z/\sqrt{\frac1d\sum_i z_i^2+\epsilon}$, then
$$w=f(\hat z),\qquad f:\mathcal Z\to\mathcal W,$$
an 8-layer MLP (4 here). The intermediate space $\mathcal W$ need not match the
fixed prior of $\mathcal Z$, so it can **unwarp** the data manifold and
disentangle factors of variation.

**AdaIN (adaptive instance normalization).** For a feature map $x_i$ of channel
$i$, normalize per-instance then apply a style-derived affine transform:
$$\mathrm{AdaIN}(x_i, w)=\gamma_i(w)\,\frac{x_i-\mu(x_i)}{\sigma(x_i)}+\beta_i(w),$$
where $\mu, \sigma$ are computed over the spatial dimensions of that instance and
$(\gamma(w),\beta(w))=A\,w$ is a learned affine map ("style"). Each layer
**discards the previous statistics** (via the normalization) and overwrites them
with the style, which is why a single $w$ exerts global, layer-localized control.
(In code we predict a residual scale $1+\gamma$ for stability.)

**Noise injection.** Before activation, add scaled noise:
$$x \leftarrow x + \psi\odot n,\qquad n\sim\mathcal N(0,I)\text{ per pixel},$$
with a learned per-channel scale $\psi$. This models stochastic detail
independently of the style, freeing $w$ to encode high-level structure.

**Synthesis layer.** Each block is
$$x \leftarrow \mathrm{LReLU}\big(\mathrm{AdaIN}(\,\text{noise}(\mathrm{conv}(\mathrm{up}(x)))\,,\,w)\big).$$
Training still uses the **non-saturating GAN loss**
$\mathcal L_G=-\mathbb E_z[\log D(G(z))]$, with $D$ minimizing the usual BCE.

**Why it differs from vanilla GAN.** The *objective* is unchanged; the
*generator design* changes — latent disentanglement ($\mathcal W$), per-layer
AdaIN style modulation, a learned constant input, and noise injection — yielding
controllable, higher-quality synthesis.

## 4. Generator / key component

In [ ]:
# ===== actual implementation from stylegan.py =====
from __future__ import annotations

import numpy as np

import torch

import torch.nn as nn

SEED = 0

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def make_images(n: int = 256, size: int = 16, seed: int = SEED) -> np.ndarray:
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:size, 0:size].astype(np.float32)
    cx = cy = (size - 1) / 2
    radial = -((xx - cx) ** 2 + (yy - cy) ** 2)
    radial = radial / radial.min()
    imgs = np.empty((n, 1, size, size), dtype=np.float32)
    for i in range(n):
        img = 0.3 * radial.copy()
        c = rng.integers(0, 4)
        h = size // 3
        ys = 0 if c < 2 else size - h
        xs = 0 if c % 2 == 0 else size - h
        img[ys:ys + h, xs:xs + h] += 0.7
        img += 0.03 * rng.standard_normal((size, size)).astype(np.float32)
        imgs[i, 0] = np.clip(img, 0, 1)
    return (imgs * 2 - 1).astype(np.float32)

class MappingNetwork(nn.Module):
    def __init__(self, z_dim: int = 32, w_dim: int = 32, layers: int = 4):
        super().__init__()
        net = []
        d = z_dim
        for _ in range(layers):
            net += [nn.Linear(d, w_dim), nn.LeakyReLU(0.2, True)]
            d = w_dim
        self.net = nn.Sequential(*net)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        # Normalize z onto the hypersphere (PixelNorm), as in StyleGAN.
        z = z * torch.rsqrt(z.pow(2).mean(dim=1, keepdim=True) + 1e-8)
        return self.net(z)

class NoiseInjection(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.weight = nn.Parameter(torch.zeros(1, channels, 1, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        noise = torch.randn(x.size(0), 1, x.size(2), x.size(3), device=x.device)
        return x + self.weight * noise

class AdaIN(nn.Module):
    def __init__(self, w_dim: int, channels: int):
        super().__init__()
        self.affine = nn.Linear(w_dim, channels * 2)  # -> (scale, bias) per channel
        self.channels = channels

    def forward(self, x: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
        style = self.affine(w).view(x.size(0), 2, self.channels, 1, 1)
        scale, bias = style[:, 0], style[:, 1]
        mu = x.mean(dim=(2, 3), keepdim=True)
        sigma = x.std(dim=(2, 3), keepdim=True) + 1e-8
        return (1 + scale) * (x - mu) / sigma + bias

class SynthesisBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, w_dim: int, upsample: bool):
        super().__init__()
        self.upsample = upsample
        self.conv = nn.Conv2d(in_ch, out_ch, 3, 1, 1)
        self.noise = NoiseInjection(out_ch)
        self.adain = AdaIN(w_dim, out_ch)
        self.act = nn.LeakyReLU(0.2, True)

    def forward(self, x: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
        if self.upsample:
            x = nn.functional.interpolate(x, scale_factor=2, mode="nearest")
        x = self.conv(x)
        x = self.noise(x)
        x = self.adain(x, w)
        return self.act(x)

class StyleGenerator(nn.Module):
    def __init__(self, z_dim: int = 32, w_dim: int = 32, ch: int = 32, img_ch: int = 1):
        super().__init__()
        self.z_dim = z_dim
        self.mapping = MappingNetwork(z_dim, w_dim)
        # Learned constant input (StyleGAN starts synthesis from a constant tensor).
        self.const = nn.Parameter(torch.randn(1, ch, 4, 4))
        self.adain0 = AdaIN(w_dim, ch)
        self.noise0 = NoiseInjection(ch)
        self.block1 = SynthesisBlock(ch, ch, w_dim, upsample=True)   # 4 -> 8
        self.block2 = SynthesisBlock(ch, ch, w_dim, upsample=True)   # 8 -> 16
        self.to_img = nn.Sequential(nn.Conv2d(ch, img_ch, 1), nn.Tanh())

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        w = self.mapping(z)
        x = self.const.expand(z.size(0), -1, -1, -1)
        x = self.adain0(self.noise0(x), w)         # style the constant input
        x = self.block1(x, w)
        x = self.block2(x, w)
        return self.to_img(x)

## 5. Trainer / losses

In [ ]:
# ===== actual implementation from stylegan.py =====
class Discriminator(nn.Module):
    def __init__(self, ndf: int = 32, img_ch: int = 1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(img_ch, ndf, 4, 2, 1), nn.LeakyReLU(0.2, True),       # 16->8
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1), nn.LeakyReLU(0.2, True),      # 8->4
            nn.Conv2d(ndf * 2, 1, 4, 1, 0),                                # ->1 logit
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).view(x.size(0), 1)

class StyleGANTorch:
    def __init__(self, z_dim: int = 32, lr: float = 2e-4):
        torch.manual_seed(SEED)
        self.dev = get_device()
        self.z_dim = z_dim
        self.G = StyleGenerator(z_dim).to(self.dev)
        self.D = Discriminator().to(self.dev)
        self.optG = torch.optim.Adam(self.G.parameters(), lr=lr, betas=(0.0, 0.99))
        self.optD = torch.optim.Adam(self.D.parameters(), lr=lr, betas=(0.0, 0.99))
        self.bce = nn.BCEWithLogitsLoss()

    def fit(self, real: np.ndarray, steps: int = 250, batch: int = 32):
        real = torch.as_tensor(real, dtype=torch.float32, device=self.dev)
        self.d_hist, self.g_hist = [], []
        ones = torch.ones(batch, 1, device=self.dev)
        zeros = torch.zeros(batch, 1, device=self.dev)
        for _ in range(steps):
            x = real[torch.randint(0, len(real), (batch,), device=self.dev)]
            # --- D step ---
            z = torch.randn(batch, self.z_dim, device=self.dev)
            fake = self.G(z).detach()
            lossD = self.bce(self.D(x), ones) + self.bce(self.D(fake), zeros)
            self.optD.zero_grad(); lossD.backward(); self.optD.step()
            # --- G step (non-saturating) ---
            z = torch.randn(batch, self.z_dim, device=self.dev)
            lossG = self.bce(self.D(self.G(z)), ones)
            self.optG.zero_grad(); lossG.backward(); self.optG.step()
            self.d_hist.append(lossD.item()); self.g_hist.append(lossG.item())
        return self

    @torch.no_grad()
    def generate(self, n: int) -> np.ndarray:
        self.G.eval()
        z = torch.randn(n, self.z_dim, device=self.dev)
        out = self.G(z).cpu().numpy()
        self.G.train()
        return out

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    real = make_images(256)
    print(f"data: {real.shape}, range [{real.min():.2f}, {real.max():.2f}]")

    gan = StyleGANTorch()
    # Fast forward pass sanity check (the style-based synthesis mechanism).
    sample = gan.generate(4)
    print(f"forward pass OK: generated {sample.shape}, range "
          f"[{sample.min():.2f}, {sample.max():.2f}]")

    gan.fit(real, steps=250, batch=32)
    print(f"D loss trend: {np.mean(gan.d_hist[:30]):.3f} -> {np.mean(gan.d_hist[-30:]):.3f}")
    print(f"G loss trend: {np.mean(gan.g_hist[:30]):.3f} -> {np.mean(gan.g_hist[-30:]):.3f}")

    fake = gan.generate(64)
    print(f"real pixel mean={real.mean():.3f} std={real.std():.3f}")
    print(f"fake pixel mean={fake.mean():.3f} std={fake.std():.3f} "
          f"(approaching real => style synthesis is learning structure)")

## 6. Train

In [ ]:
demo()

## 7. Visualization

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
import stylegan as M

real = M.make_images(256)
gan = M.StyleGANTorch().fit(real, steps=150, batch=32)  # lighter retrain just for the picture
fake = gan.generate(8)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for j in range(8):
    axes[0, j].imshow(real[j, 0], cmap="gray", vmin=-1, vmax=1); axes[0, j].axis("off")
    axes[1, j].imshow(fake[j, 0], cmap="gray", vmin=-1, vmax=1); axes[1, j].axis("off")
axes[0, 0].set_title("real", loc="left"); axes[1, 0].set_title("style-generated", loc="left")
fig.suptitle("StyleGAN (simplified): real (top) vs style-based synthesis (bottom)")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- StyleGAN keeps the GAN game but redesigns $G$: $z\to w$ mapping, AdaIN style
  modulation at every layer, a learned constant input, and noise injection.
- AdaIN is the workhorse — normalize, then overwrite statistics with a
  $w$-derived scale/bias, giving layer-localized control over attributes.
- Pitfalls: AdaIN's normalization can create "blob" artifacts (StyleGAN2 replaces
  it with weight demodulation); the mapping network needs enough depth to
  disentangle; noise injection helps texture but can dominate if its scale grows.